# Prolog Program Explainer using LangChain

This notebook creates a tool that provides natural language explanations of Prolog programs using LangChain.

In [1]:
# Import necessary libraries
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import StrOutputParser
from langchain_ollama import OllamaLLM
import shutil

# Load environment variables (especially API keys)
# load_dotenv()

In [2]:
ollama_exists = shutil.which('ollama') is not None

if not ollama_exists:
    # Install Ollama (if not already installed)
    !curl -fsSL https://ollama.com/install.sh | sh

    # Start the Ollama server
    !ollama serve

In [3]:
# Define the output structure with a Pydantic model
class PrologExplanation(BaseModel):
    summary: str = Field(description="A brief summary of what the Prolog program does")
    primitive: List[dict] = Field(description="List of dependent primitives and their descriptions")
    explanation: str = Field(description="How the program logic works")
    
def explain_prolog(llm, prompt_template, prolog_code, variables=['prolog_code']):
    """Generate a natural language explanation of a Prolog program.
    
    Args:
        llm: The language model to use
        prompt: The prompt template to use
        prolog_code (str): The Prolog code to explain
        
    Returns:
        PrologExplanation: Structured explanation of the Prolog program
    """
    # Initialize the output parser
    parser = PydanticOutputParser(pydantic_object=PrologExplanation)
    
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=variables,
        partial_variables={"JSON_format": parser.get_format_instructions()}
    )
    
    # Get raw string response from the LLM
    chain = prompt | llm | StrOutputParser()
    raw_result = chain.invoke({"prolog_code": prolog_code})
    
    print(f"===== Generated Raw Text =====\n{raw_result}")
    
    # Parse the raw string into a structured PrologExplanation object
    try:
        # Try to parse the output directly
        parsed_result = parser.parse(raw_result)
        
        # print(f"\n===== Parsed Prolog Explanation =====\n{parsed_result}")
        print(f"\n===== Parsed Text =====")

        print(f"Summary: {parsed_result.summary}\n")
        print("Primitives:")
        for pred in parsed_result.primitive:
            print(f"- {pred.get('name')}: {pred.get('description')}")
        print(f"\nExplanation: {parsed_result.explanation}\n")
        return parsed_result
    except Exception as e:
        # If direct parsing fails, attempt to extract structured data from text
        print(f"Parsing error: {e}")
        print("Returning raw output instead. Check format of LLM response.")
        return raw_result

In [4]:
# Gemma 3 4B
llm = OllamaLLM(
    model = 'gemma3:4b',
    temperature = 0
)

In [5]:
explanation_template = """
You are an expert Prolog interpreter who explains Prolog code to beginners.

I. Analyze the following Prolog program which is divided into three sections:
1. Target rules: The main predicates that define the goal of the program
2. Primitives: Supporting predicates that help implement the target rules
3. Biases: Input and output signature directions for the primitives

{prolog_code}

II. Your response MUST follow the below JSON format EXACTLY with no additional text:

{JSON_format}

YOUR EXPLANATION MUST:

In "summary": Provide a 1-2 sentence overview of what the entire Prolog program accomplishes.
In "primitive": List each primitive from Target rules as dictionaries with "name" and "description" keys.
In "explanation": Explain how the Target rules use the Primitives to achieve the program's purpose.

IMPORTANT FORMATTING RULES:

Return ONLY valid parseable JSON with the exact field names shown above.
Ensure all strings are properly escaped for valid JSON.
DO NOT return JSON schema information or "properties" fields.
Do not include markdown formatting symbols, code blocks, or explanatory text outside the JSON structure.
"""

## Example Usage

In [6]:
# Example Prolog code - Family relationships
family_prolog = """
%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
parent(X, Y) :- father(X, Y).
parent(X, Y) :- mother(X, Y).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
# direction(grandparent,(in,out)).
direction(parent,(in,out)). 
direction(father,(in,out)). 
direction(mother,(in,out)). 
"""

explanation = explain_prolog(llm, explanation_template, family_prolog)

===== Generated Raw Text =====
```json
{
  "summary": "This Prolog program defines a grandparent relationship between individuals based on parent-child relationships. It determines if X is a grandparent of Z if X has a parent Y, and Y is a parent of Z.",
  "primitive": [
    {
      "name": "grandparent",
      "description": "Determines if X is a grandparent of Z based on parent relationships."
    },
    {
      "name": "parent",
      "description": "Defines parent relationships through father and mother predicates."
    },
    {
      "name": "father",
      "description": "Represents the father relationship between two individuals."
    },
    {
      "name": "mother",
      "description": "Represents the mother relationship between two individuals."
    }
  ],
  "explanation": "The `grandparent(X, Z)` rule uses the `parent(X, Y)` and `parent(Y, Z)` rules.  First, it checks if X has a parent Y (using `parent(X, Y)`). Then, it verifies if Y is a parent of Z (using `parent(Y, Z)`). 

## Custom Prolog Code Explanation

Without code comments

In [7]:
# Example Prolog code - List processing
list_prolog = """

%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

inv_ho_0(A,B):- same_circuit(A,B),subpath(B,A).
inv_ho_1(A,B):- same_circuit(A,B),not(subpath(B,A)).

partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
all(P, X, L):- 
    findall(H, call(P, X, H), L).

has_branches(X) :- is_connected(X, Y), is_connected(X, Z), Y \\== Z.

subpath(X, X).
subpath(X, Y) :- is_connected(X, Z),  not(has_branches(X)), subpath(Z, Y).
not_subpath(X, Y) :- not(subpath(X, Y)).

same_circuit(X, Y) :- gate(X), gate(Y), N is X // 100, M is Y // 100, N == M.

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
direction(partition,(in,out,out)). 
direction(subpath,(in,out)).
direction(not_subpath,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).

"""

explanation = explain_prolog(llm, explanation_template, list_prolog)

===== Generated Raw Text =====
```json
{
  "summary": "This Prolog program defines a system for identifying inverted horizontal connections within a circuit diagram. It uses a set of rules to determine if a connection is inverted based on subpaths and circuit similarity.",
  "primitive": [
    {
      "name": "inv_ho_0",
      "description": "Checks if two circuit elements are the same and if one is a subpath of the other."
    },
    {
      "name": "inv_ho_1",
      "description": "Checks if two circuit elements are the same but not a subpath of each other."
    },
    {
      "name": "partition",
      "description": "Partitions a set of circuit elements into groups based on inverted horizontal connections."
    },
    {
      "name": "all",
      "description": "Applies a predicate to a list of elements and collects the results."
    },
    {
      "name": "has_branches",
      "description": "Checks if a circuit element has branches connected to it."
    },
    {
      "name": "su

With code comments

In [8]:
# Example Prolog code - List processing
list_prolog = """

%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

% Gates affected by a test and those not affected
inv_ho_1(A,B):- same_circuit(A,B),not(subpath(B,A)).
inv_ho_0(A,B):- same_circuit(A,B),subpath(B,A).

% Given Gate A find gates on the same branch as A and gates on other branches
partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
% HO predicate
all(P, X, L):- 
    findall(H, call(P, X, H), L).

% Are multiple gates connected to gate X?
has_branches(X) :- is_connected(X, Y), is_connected(X, Z), Y \\== Z.

% Is gate X to gate Y a path (no branches)?
subpath(X, X).
subpath(X, Y) :- is_connected(X, Z),  not(has_branches(X)), subpath(Z, Y).
not_subpath(X, Y) :- not(subpath(X, Y)).

% Does gate X share a subpath with gate Y (Y precedes X if X \\== Y)?
same_circuit(X, Y) :- gate(X), gate(Y), N is X // 100, M is Y // 100, N == M.

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
direction(partition,(in,out,out)). 
direction(subpath,(in,out)).
direction(not_subpath,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).

"""

explanation = explain_prolog(llm, explanation_template, list_prolog)

===== Generated Raw Text =====
```json
{
  "summary": "This Prolog program defines a system for partitioning a given gate into its constituent sub-gates, identifying branches, and determining if gates share a common circuit. It utilizes primitives to check connectivity and subpath relationships between gates.",
  "primitive": [
    {
      "name": "all",
      "description": "Applies a predicate to all elements in a list, collecting the results."
    },
    {
      "name": "has_branches",
      "description": "Checks if a gate has connections to other gates, indicating a branch."
    },
    {
      "name": "subpath",
      "description": "Determines if two gates are connected in a direct path (no branches)."
    },
    {
      "name": "not_subpath",
      "description": "Checks if two gates are not directly connected in a path."
    },
    {
      "name": "same_circuit",
      "description": "Checks if two gates belong to the same circuit, considering their numerical values."
    },
  